In [1]:
# notebook to compile all of the csvs into a single array
import os
import glob
import pandas as pd
import numpy as np

import cantera as ct


In [2]:
mech_dir = '/scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4'
# mech_dir = '/scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_MAX/RMG_MAX_2'
gas = ct.Solution(os.path.join(mech_dir, 'chem_annotated.yaml'))

In [4]:
main_table = 7
# Compile the species sensitivities if that hasn't been done yet
sp_delay_file = os.path.join(mech_dir, f'table_{main_table:04}', f'species_delays_{main_table:04}.npy')
if not os.path.exists(sp_delay_file) or True:
    sp_files = glob.glob(os.path.join(mech_dir, f'table_{main_table:04}', f'spec_delay_{main_table:04}_*.npy'))
#     N = len(sp_files)
    N = gas.n_species
    K = 51
    spec_delays = np.zeros((N, K))
    for i in range(N):
        try:  
            sp_file = os.path.join(mech_dir, f'table_{main_table:04}', f'spec_delay_{main_table:04}_{i:04}.npy')
            spec_delays[i, :] = np.load(sp_file)
        except FileNotFoundError:
            print(f'couldnt find {sp_file}')
    np.save(os.path.join(mech_dir, f'table_{main_table:04}', f'species_delays_{main_table:04}.npy'), spec_delays)
else:
    spec_delays = np.load(sp_delay_file)

In [27]:
N

374

In [5]:
# load examples to get the right size
test_sp_file = os.path.join(mech_dir, 'table_0007', 'spec_delay_0007_0000.npy')
test_rxn_file = os.path.join(mech_dir, 'table_0007', 'reaction_delays_0007_0000.npy')

N_REACTIONS_PER_FILE = 10

K = 51
N = spec_delays.shape[0]
M = np.load(test_rxn_file).shape[0]
print(f'N={N}', 'species')
print(f'M={M}', 'reactions')

all_delays_ever = np.zeros((N + M, 12 * K))

N=381 species
M=3002 reactions


In [20]:
spec_delays.shape

(369, 51)

In [6]:
# compile everything into a humongous array

#             table1 table2 ... table12
# species 1
# species 2
# .........
# species N
# reaction 1
# reaction 2
# .........
# reaction M


# for table_index in range(1, 13):
for table_index in [7]:
    table_dir = os.path.join(mech_dir, f'table_{table_index:04}')
    
    rxn_files = glob.glob(os.path.join(table_dir, f'reaction_delays_{table_index:04}_*.npy'))

    all_delays_ever[0:N, (table_index - 1) * K: table_index * K] = spec_delays
    
    
    # fill in the reaction files
    rxn_table = np.zeros((M, K))
    for i in range(0, int(3500 / N_REACTIONS_PER_FILE)):
        rxn_delay_file = os.path.join(table_dir, f'reaction_delays_{table_index:04}_{i * N_REACTIONS_PER_FILE:04}.npy')
        if not os.path.exists(rxn_delay_file):
            print('missing: ', i, rxn_delay_file)
            continue  # TODO use assert and do not continue
        rxn_table += np.load(rxn_delay_file)
    all_delays_ever[N:, (table_index - 1) * K: table_index * K] = rxn_table

missing:  7 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0070.npy
missing:  8 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0080.npy
missing:  9 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0090.npy
missing:  10 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0100.npy
missing:  11 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0110.npy
missing:  12 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0120.npy
missing:  13 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/reaction_delays_0007_0130.npy
missing:  14 /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0007/r

In [15]:
all_delays_ever.shape

(3400, 612)

In [7]:
# save the resulting delay array
np.save(os.path.join(mech_dir, 'total_perturbed_mech_delays.npy'), all_delays_ever)

In [13]:
np.load(rxn_delay_file)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
all_delays_ever.shape

(1368, 612)

In [8]:
# Also compile the base delays into a giant 1 x (12 * K) array
total_base_delays = np.zeros(12 * K)
for table_index in range(1, 13):
    table_dir = os.path.join(mech_dir, f'table_{table_index:04}')
    base_delay_file = os.path.join(table_dir, f'base_delays_{table_index:04}.npy')
    
    if not os.path.exists(base_delay_file):
        print(f'Missing base delay file {base_delay_file}')
        continue
        raise OSError(f'Missing base delay file {base_delay_file}')
    
    total_base_delays[(table_index - 1) * K:table_index * K] = np.load(base_delay_file)


Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0001/base_delays_0001.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0002/base_delays_0002.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0003/base_delays_0003.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0004/base_delays_0004.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0005/base_delays_0005.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0006/base_delays_0006.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RMG_min/RMG_min_4/table_0008/base_delays_0008.npy
Missing base delay file /scratch/harris.se/guassian_scratch/advanced_autoscience/RM

In [9]:
# save the resulting base delay array
np.save(os.path.join(mech_dir, 'total_base_delays.npy'), total_base_delays)

In [12]:
415 / 51

8.137254901960784

In [13]:
415 % 51

7

In [ ]:
# see how many species calcs zero
zeros = 0
cols = set()
rows = set()
for i in range(0, 130):
    for j in range(all_delays_ever.shape[1]):
#         if j == 415:
#             continue
        if all_delays_ever[i, j] == 0:
            print(i, j)
#             cols.add(j)
#             if j != 415:
#                 rows.add(i)
#             zeros += 1
# print(zeros, '/', 130 * all_delays_ever.shape[1])
# print(cols)
# print(rows)

In [ ]:
all_delays_ever[67, :]

In [ ]:
# see how many rows are completely zero
zero_rows = 0
zero_row_set = set()
for i in range(0, all_delays_ever.shape[0]):
    if np.sum(all_delays_ever[i,:]) == 0:
        zero_rows += 1
        zero_row_set.add(i)
print(zero_rows, '/', all_delays_ever.shape[0])

In [ ]:
# print out what's missing - for debugging
for table_index in range(1, 13):
    table_dir = os.path.join(mech_dir, f'table_{table_index:04}')
    
    for i in range(0, 51):
        delay_file = os.path.join(table_dir, f'reaction_delays_{table_index:04}_{i * 50:04}.npy')
        if not os.path.exists(delay_file):
            print('missing: ', i, delay_file[-50:])

In [ ]:
for j in range(0, all_delays_ever.shape[1]):
    for i in range(0, all_delays_ever.shape[0]):
        if all_delays_ever[i, j] == 0 and i not in zero_row_set:
            print(f'({i}, {j}) is blank\ttable {int(j / 51) + 1}\tblock {int((i - 130) / 50)}')

In [ ]:
# count total zeros
np.sum(all_delays_ever == 0)

In [ ]:
668 * 51 * 12

In [ ]:
all_delays_ever[:, 415]

In [ ]:
# /work/westgroup/harris.se/autoscience/reaction_calculator/delay_uncertainty/base_rmg_1week/chem_annotated.inp

In [23]:
delays0 = np.load('/scratch/harris.se/guassian_scratch/RMG_min/RMG_min_2/table_0007/spec_delay_0007_0000.npy')
delays1 = np.load('/scratch/harris.se/guassian_scratch/RMG_min/RMG_min_2/table_0007/spec_delay_0007_0001.npy')

In [27]:
delays0 - delays1

array([ 1.33920255e-10, -1.51799368e-10,  3.30035235e-11,  1.63495200e-10,
       -6.27288031e-11,  6.43710748e-10,  2.56606724e-11,  5.38174781e-10,
        5.25737000e-10, -6.42005243e-10, -4.33118547e-10,  4.19786934e-10,
       -7.29999573e-10,  8.78640372e-11,  1.91159080e-10, -2.54295387e-10,
       -3.35783571e-10,  1.20110569e-10, -2.85912878e-10,  4.23862855e-11,
        1.18942554e-10,  6.69812792e-10,  4.59545353e-10, -5.57187574e-10,
        2.18803526e-10,  2.90170318e-11, -6.60415873e-10,  2.50100322e-10,
        2.83487957e-10, -5.05671893e-15,  2.63640863e-11,  3.43381445e-10,
        6.18390790e-11, -3.07201545e-10, -2.26823922e-10, -4.44880304e-10,
       -3.68392955e-11,  9.09025806e-11,  3.94597806e-10, -4.89408462e-11,
        4.85815112e-11,  1.24073789e-10, -1.30818533e-11,  9.25102078e-11,
        2.26287639e-10,  2.34895168e-10,  1.27591996e-10,  3.78032010e-10,
       -6.52017910e-11, -9.68622925e-11,  9.29015675e-11])

In [25]:
delays1

array([0.01361021, 0.01128721, 0.00960286, 0.00840422, 0.00758026,
       0.00700688, 0.00675298, 0.00664738, 0.00670083, 0.00688926,
       0.00719414, 0.0076002 , 0.00809384, 0.00866174, 0.00928983,
       0.00996249, 0.01066211, 0.01136912, 0.01206267, 0.01272223,
       0.01333046, 0.01387688, 0.01436021, 0.0147843 , 0.01514295,
       0.01540312, 0.01551288, 0.01543433, 0.01516459, 0.01473006,
       0.01417004, 0.01352383, 0.01282452, 0.01209749, 0.01136122,
       0.01062879, 0.00990936, 0.00920941, 0.00853358, 0.00788522,
       0.00726679, 0.00668004, 0.00612613, 0.0056057 , 0.00511892,
       0.00466553, 0.0042449 , 0.00385607, 0.0034978 , 0.00316865,
       0.00286709])